In [ ]:
import os, sys
sys.path.insert(0, os.environ.get("GMS_RAG_TUTORIAL", os.path.dirname(os.getcwd())))
from book_kit import load_store, CORPUS, STORE, EVAL_COHORT, DEVICE, store_config

# Ch13 — Evaluating a RAG system

A single accuracy number hides the failures that matter for a grounded
system. We measure **trust**: did paraphrases bind, did exact numbers come
back byte-exact, did the system *abstain* on prose it has no triples for,
and — the release gate — were there **zero confident-wrong** answers?

This chapter runs the shipped `evaluate` harness over `data/eval_cohort.json`,
then derives the trust metrics the harness does not compute directly
(provenance rate, abstention precision, confident-wrong count) from the
per-case `RagAnswer` records the pipeline already returns.

## 1. The labeled cohort

Twelve questions across the types from F2: factual, numeric/exact,
multi-hop, paraphrase, unanswerable (prose → abstain), and one
**adversarial** case whose answer must equal the ENM value 120.0.

In [ ]:
import json, pathlib
from knowlytix.knowledge.rag import EvalCase, evaluate

COHORT = pathlib.Path(EVAL_COHORT)
rows = json.loads(COHORT.read_text())
cases = [
    EvalCase(
        question=r["question"],
        expected_answer=r.get("expected_answer"),
        expect_decision=r.get("expect_decision"),
    )
    for r in rows
]
print(f"{len(cases)} cases; types:",
      sorted({r["type"] for r in rows}))

Expected: `12 cases; types: ['adversarial', 'factual', 'multi_hop',
'numeric', 'paraphrase', 'unanswerable']`.

## 2. A deterministic backend (CI determinism)

Evaluation must be reproducible. For CI we inject a scripted
`LLMBackend` that maps the cohort's questions to the extraction/synthesis
behavior we expect from a *correctly* grounded run. The real Qwen path is
shown in cell 5 and run unscripted by the lead. Both implement the same
`LLMBackend` ABC (`call`, `model_name`).

In [ ]:
from knowlytix.knowledge.llm_backend import LLMBackend


class ScriptedBackend(LLMBackend):
    """Deterministic backend: a dict of (substring -> reply) rules.

    Used ONLY to make CI deterministic. It is NOT part of the library and
    must never gate a release decision in production -- the real Qwen run
    does. See cell 5 for the live path.
    """

    def __init__(self, rules: dict[str, str], default: str = ''):
        self._rules = rules
        self._default = default

    def call(self, system: str, user: str, max_tokens: int = 2048) -> str:
        for needle, reply in self._rules.items():
            if needle.lower() in user.lower():
                return reply
        return self._default

    @property
    def model_name(self) -> str:
        return 'scripted-ci'

## 3. Wire the pipeline

Load the trained store and build a `RagPipeline`. The accept threshold is
the value Ch12 calibrated; `verify_llm_output=True` turns on the GMS
self-verifier (Ch10) so a confident-wrong answer is caught before it is
accepted. **Lead executes — needs the store; no Qwen here (scripted).**

In [ ]:
# [LEAD: execute in CI -- loads the trained store]
from knowlytix.knowledge.rag import RagConfig, RagPipeline

store = load_store()   # exact build config (GeometryConfig(64,64,32,32), cap)

# Extraction rules sufficient for the cohort's grounded questions.
extract_rules = {
    "cloud platform revenue": "cloud platform | has_revenue | ?",
    "total revenue": "total | has_revenue | ?",
    "work in retail": "retail | has_headcount | ?",
    "net income": "net income | has_fy2025 | ?",
    "shareholders equity": "shareholders equity | has_value | ?",
    "who is the ceo": "northwind industries | has_head | ?",
    "region runs the division that contains cloud platform":
        "cloud platform | has_division | ?x\n?x | has_region | ?",
    "who heads the division that contains logistics":
        "logistics | has_division | ?x\n?x | has_head | ?",
    "topline": "total | has_revenue | ?",
}
extract_llm = ScriptedBackend(extract_rules, default='')
# Synthesis just restates the retrieved fact's tail (grounded-only).
synth_llm = ScriptedBackend({}, default='See grounded facts.')

cfg = RagConfig(
    llm=synth_llm,
    llm_extract=extract_llm,
    binding='embedding',
    verify_llm_output=True,
    on_verify_fail='abstain',
    accept_threshold=0.0,   # Ch12-calibrated value goes here
    relevance_gate=False,   # scripted backend can't judge relevance
)
pipe = RagPipeline.from_store(store, cfg)

## 4. The report a release gates on

`evaluate` returns an `EvalReport` with bind rate, answer accuracy,
decision accuracy, and accept rate. We then walk each case's `RagAnswer`
to add the three trust metrics the report omits — and the one that blocks
a release: **confident-wrong = 0**.

In [ ]:
# [LEAD: execute in CI]
report = evaluate(pipe, cases)
print('EvalReport:', report.as_dict())

Expected (grounded run): `bind_rate` near 1.0 on the bindable subset,
`answer_accuracy` 1.0 on the cases carrying an `expected_answer`, and
`decision_accuracy` 1.0 (the two prose questions abstain).

In [ ]:
# [LEAD: execute in CI] -- derive trust metrics from per-case answers.
def trust_metrics(pipe, rows, cases):
    confident_wrong = 0
    with_prov = answered = 0
    abst_correct = abst_total = 0
    for row, case in zip(rows, cases):
        ans = pipe.query(case.question)
        accepted = ans.decision == 'accept'
        # provenance rate: accepted answers must carry a source span.
        if accepted:
            answered += 1
            if any(f.location for f in ans.sources):
                with_prov += 1
        # abstention precision: of cases we abstained on, how many SHOULD
        # have abstained?
        if ans.decision == 'abstain':
            abst_total += 1
            if case.expect_decision == 'abstain':
                abst_correct += 1
        # confident-wrong: accepted AND the expected answer is missing.
        if accepted and case.expected_answer is not None:
            exp = case.expected_answer
            hit = exp in ans.answer or any(
                exp == f.tail for f in ans.sources)
            if not hit:
                confident_wrong += 1
        # accepted on a question that should abstain is also confident-wrong.
        if accepted and case.expect_decision == 'abstain':
            confident_wrong += 1
    return {
        'provenance_rate': with_prov / answered if answered else 0.0,
        'abstention_precision': abst_correct / abst_total if abst_total else 1.0,
        'confident_wrong': confident_wrong,
    }

trust = trust_metrics(pipe, rows, cases)
print('Trust:', trust)

Expected: `provenance_rate` 1.0 (every accepted answer points to a table
cell, e.g. `:15:553-558` for Cloud Platform revenue), `abstention_precision`
1.0 (the Outlook and Risk-Factors prose questions are the only abstentions,
and both *should* abstain), and **`confident_wrong` 0**.

## 5. Abstention precision on the unanswerable subset

The prose sections (MD&A, Risk Factors, Outlook) carry **no triples** —
coverage_ratio is 0.56 (Ch11). A trustworthy system abstains there rather
than paraphrasing prose into a confident answer. We isolate the two
`unanswerable` cases and confirm both abstain.

In [ ]:
# [LEAD: execute in CI]
prose = [(r, c) for r, c in zip(rows, cases)
         if r['type'] == 'unanswerable']
for r, c in prose:
    a = pipe.query(c.question)
    print(f"{r['id']}: decision={a.decision!r} notice={a.notice!r}")
abstained = all(pipe.query(c.question).decision == 'abstain' for _, c in prose)
print('all prose abstained:', abstained)

Expected: both `q-outlook` and `q-risks` print `decision='abstain'` with a
no-binding / no-match notice. Abstention precision on this subset is 1.0.

## 6. The real evaluation run (Qwen)

In production the gate runs against local Qwen2.5-3B-Instruct, not the
scripted backend. Same `evaluate` call, same `trust_metrics`. **Lead runs
this unscripted in CI; it needs the GPU.**

In [ ]:
# [LEAD: execute in CI on GPU -- the real gate]
from knowlytix.knowledge.geode import QWEN_3B
from knowlytix.knowledge.llm_backend import LocalTransformersBackend

qwen = LocalTransformersBackend(QWEN_3B)
qwen_cfg = RagConfig(
    llm=qwen, binding='embedding',
    verify_llm_output=True, on_verify_fail='abstain',
)
qwen_pipe = RagPipeline.from_store(store, qwen_cfg)
qwen_report = evaluate(qwen_pipe, cases)
print('Qwen EvalReport:', qwen_report.as_dict())
print('Qwen Trust:', trust_metrics(qwen_pipe, rows, cases))

## Exercise — an adversarial confident-wrong case

Add a case whose *grounded* answer is correct (120.0) but force a draft
that asserts the wrong number, and confirm the self-verifier (Ch10) turns
the would-be accept into an abstain — keeping `confident_wrong` at 0.

Here we use a synthesis backend that *lies* (claims 999.0 for Cloud
Platform revenue). The GMS self-verifier decomposes the draft into claim
triples, finds `cloud platform has_revenue 999.0` contradicts the asserted
120.0, and `on_verify_fail='abstain'` blocks acceptance.

In [ ]:
# [LEAD: execute in CI] -- worked solution.
lying_synth = ScriptedBackend(
    {'cloud platform': 'Cloud Platform revenue was 999.0.'},
    default='See grounded facts.')
adv_cfg = RagConfig(
    llm=lying_synth, llm_extract=extract_llm,
    binding='embedding', verify_llm_output=True,
    on_verify_fail='abstain', relevance_gate=False,
)
adv_pipe = RagPipeline.from_store(store, adv_cfg)
adv = adv_pipe.query('What is Cloud Platform revenue?')
print('decision:', adv.decision)
print('verification ok:', adv.verification.get('ok'))
print('notice:', adv.notice)
# The lie never reaches the user as an accepted answer.
assert adv.decision == 'abstain'
assert adv.verification.get('ok') is False

## Self-check — zero confident-wrong on the adversarial subset

The chapter's claim: a trustworthy RAG eval reports **zero confident-wrong**.
We assert it on the adversarial subset of the cohort.

In [ ]:
# [LEAD: execute in CI]
adv_rows = [(r, c) for r, c in zip(rows, cases)
            if r['type'] == 'adversarial']
adv_trust = trust_metrics(adv_pipe,
                          [r for r, _ in adv_rows],
                          [c for _, c in adv_rows])
assert adv_trust['confident_wrong'] == 0, adv_trust
print('PASS: zero confident-wrong on the adversarial subset:', adv_trust)